In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
columns_with_nan = df.columns[df.isnull().any()].tolist()
if 'Target' in columns_with_nan:
    columns_with_nan.remove('Target')

print(f"Columns with missing values (excluding 'Target'): {columns_with_nan}")

for col in columns_with_nan:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

print("Missing values after median imputation:")
print(df[columns_with_nan].isnull().sum().sum())

for col in columns_with_nan:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print("Missing values after median imputation:")
print(df[columns_with_nan].isnull().sum().sum())

In [ ]:
# Task 2: Write your code here:
print(f"Number of duplicate rows before removal: {df.duplicated().sum()}")

In [ ]:
# Task 3: Write your code here:
import pandas as pd

d_columns_int = [col for col in df.columns if col.startswith('D_') and df[col].dtype == 'int64']

print(f"'D_'-prefixed integer columns: {d_columns_int}")

categorical_d_columns = []
threshold_unique_values = min(50, int(len(df) * 0.05)) # Example: max 50 unique values or 5% of total rows

print("Candidate 'D_'-prefixed integer columns and their unique value counts:")
for col in d_columns_int:
    unique_count = df[col].nunique()
    print(f"  Column '{col}': {unique_count} unique values")
    if unique_count <= threshold_unique_values:
        categorical_d_columns.append(col)

print(f"\nColumns identified as categorical (unique values <= {threshold_unique_values}): {categorical_d_columns}")

for col in categorical_d_columns:
    df[col] = df[col].astype('category')

print("Data types of identified categorical 'D_'-prefixed columns after conversion:")
print(df[categorical_d_columns].dtypes)

In [ ]:
# Task 4: Write your code here:
import numpy as np
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

if 'Target' in numerical_cols:
    numerical_cols.remove('Target')

numerical_features_for_scaling = df.select_dtypes(include=np.number).columns.tolist()
if 'Target' in numerical_features_for_scaling:
    numerical_features_for_scaling.remove('Target')

print(f"Numerical columns to be scaled (excluding 'Target'): {numerical_features_for_scaling}")

scaler = StandardScaler()
df[numerical_features_for_scaling] = scaler.fit_transform(df[numerical_features_for_scaling])

print("Numerical features after scaling (first 5 rows):")
print(df[numerical_features_for_scaling].head())

In [ ]:
# Task 5: Write your code here:
target_counts = df['Target'].value_counts()
target_percentages = df['Target'].value_counts(normalize=True) * 100

print("Target variable value counts:\n", target_counts)
print("\nTarget variable percentages:\n", target_percentages)

if target_percentages.min() < 30:
    print("\nConclusion: The target variable 'Target' appears to be imbalanced.")
else:
    print("\nConclusion: The target variable 'Target' appears to be balanced.")

In [ ]:
# Task 1: Write your code here:
X = df.drop('Target', axis=1)
y = df['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
import numpy as np

categorical_features_indices = [X.columns.get_loc(col) for col in X.select_dtypes(include='category').columns]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = CatBoostClassifier(
        iterations=100,
        learning_rate=0.1,
        depth=6,
        l2_leaf_reg=3,
        loss_function='Logloss',
        eval_metric='F1',
        random_seed=42,
        verbose=0,
        cat_features=categorical_features_indices
    )

    model.fit(X_train, y_train, early_stopping_rounds=10, eval_set=(X_val, y_val), verbose=0)
    y_pred = model.predict(X_val)
    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)
    print(f"Fold {fold+1} F1 Score: {f1:.4f}")

print(f"\nAveraged F1 Score across all folds: {np.mean(f1_scores):.4f}")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

feature_importances = model.get_feature_importance()
feature_names = X.columns

importance_df = pd.Series(feature_importances, index=feature_names)

sorted_importance_df = importance_df.sort_values(ascending=False)

plt.figure(figsize=(12, 8))
sorted_importance_df.head(20).plot(kind='barh')
plt.title('Top 20 Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:
golden_feature = sorted_importance_df.index[0]
print(f"The 'golden feature' is: {golden_feature}")

In [ ]:
# Task Bonus: Write your code here:
X_golden = X[[golden_feature]]

print(f"Shape of new features (X_golden) with only '{golden_feature}': {X_golden.shape}")
print(f"First 5 rows of X_golden:\n{X_golden.head()}")


In [ ]:
skf_golden = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores_golden = []

for fold, (train_index, val_index) in enumerate(skf_golden.split(X_golden, y)):
    X_train_golden, X_val_golden = X_golden.iloc[train_index], X_golden.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model_golden = CatBoostClassifier(
        iterations=100,  # Same as full model
        learning_rate=0.1,
        depth=6,
        l2_leaf_reg=3,
        loss_function='Logloss',
        eval_metric='F1',
        random_seed=42,
        verbose=0,
        cat_features=[]
    )

    model_golden.fit(X_train_golden, y_train, early_stopping_rounds=10, eval_set=(X_val_golden, y_val), verbose=0)
    y_pred_golden = model_golden.predict(X_val_golden)
    f1_golden = f1_score(y_val, y_pred_golden)
    f1_scores_golden.append(f1_golden)
    print(f"Fold {fold+1} F1 Score (Golden Feature Only): {f1_golden:.4f}")

averaged_f1_golden = np.mean(f1_scores_golden)
print(f"\nAveraged F1 Score (Golden Feature Only) across all folds: {averaged_f1_golden:.4f}")
print(f"Averaged F1 Score (Full Model) across all folds: {np.mean(f1_scores):.4f}")